# Study: Max Logarithmic Price Range Ratio — 1440-sample Rolling Window

For each asset in `payamdavaee/candles`:

1. Reads all monthly Parquet files (sorted by `ts`) via **DuckDB** from HuggingFace.
2. Applies a **1440-sample rolling window** (preceding + current row) over `vwap`.
3. For every complete window finds the **most distant** vwap from the current (right) value:
   - `max_vwap = argmax( abs(vwap − right) )` over the window
   - `log_return = ln( max_vwap / right )`  — positive when the extreme was above, negative when below
4. Reports:
   - Percentiles **P0 → P100** in steps of 10
   - Percentiles **P90 → P100** in steps of 1
   - Histogram with 100 bins

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from huggingface_hub import HfApi
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ID = "payamdavaee/candles"
ASSETS  = ["btcusdt", "ethusdt", "trumpusdt", "vineusdt", "adausdt", "xrpusdt", "dogeusdt"]
WINDOW  = 1440

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

api       = HfApi()
all_files = sorted(api.list_repo_files(REPO_ID, repo_type="dataset"))
print(f"Total files in dataset: {len(all_files)}")

In [ ]:
def get_parquet_urls(asset: str) -> list[str]:
    prefix = f"{asset}/"
    return [
        f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/{f}"
        for f in all_files
        if f.startswith(prefix) and f.endswith(".parquet")
    ]


def fetch_log_returns(asset: str, window: int = WINDOW) -> pd.Series | None:
    """
    Use DuckDB to compute ln(max_distant_vwap / current_vwap) for every
    complete 1440-sample window. Heavy lifting runs in DuckDB; only the
    log_return series is returned to Python.
    """
    urls = get_parquet_urls(asset)
    if not urls:
        return None

    files_expr = ", ".join(f"'{u}'" for u in urls)

    query = f"""
    WITH

    -- 1. Load ts + vwap only, sort chronologically
    raw AS (
        SELECT ts, vwap
        FROM read_parquet([{files_expr}])
        ORDER BY ts
    ),

    -- 2. Rolling window: 1439 preceding + current = 1440 samples
    windowed AS (
        SELECT
            vwap                                                                                   AS right_vwap,
            MAX(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW)     AS w_max,
            MIN(vwap) OVER (ORDER BY ts ROWS BETWEEN {window - 1} PRECEDING AND CURRENT ROW)     AS w_min,
            ROW_NUMBER() OVER (ORDER BY ts)                                                        AS rn
        FROM raw
    ),

    -- 3. Pick whichever extreme is farthest from current vwap (complete windows only)
    extremes AS (
        SELECT
            right_vwap,
            CASE
                WHEN ABS(w_max - right_vwap) >= ABS(w_min - right_vwap) THEN w_max
                ELSE w_min
            END AS max_vwap
        FROM windowed
        WHERE rn >= {window}
    )

    -- 4. Log return: positive = extreme was above current, negative = below
    SELECT LN(max_vwap / right_vwap) AS log_return
    FROM extremes
    WHERE right_vwap > 0 AND max_vwap > 0
    """

    return con.execute(query).df()["log_return"]

In [ ]:
pd.options.display.float_format = "{:.6f}".format

WIDE_PCTS = [i / 100 for i in range(0, 101, 10)]          # 0 % … 100 % step 10
FINE_PCTS = [i / 100 for i in range(90, 101,  1)]         # 90 % … 100 % step  1


def pct_table(s: pd.Series, pcts: list[float], label_fmt: str) -> pd.DataFrame:
    labels = [label_fmt.format(int(round(p * 100))) for p in pcts]
    return pd.DataFrame(
        {"log_return": s.quantile(pcts).values},
        index=pd.Index(labels, name="percentile"),
    )


for asset in ASSETS:
    urls = get_parquet_urls(asset)

    print(f"\n{'═' * 62}")
    print(f"  {asset.upper()}   ({len(urls)} monthly file(s))")
    print(f"{'═' * 62}")

    if not urls:
        print("  No Parquet files found — skipping.")
        continue

    lr = fetch_log_returns(asset)
    if lr is None or lr.empty:
        print("  No data returned — skipping.")
        continue

    print(f"  Windows computed : {len(lr):,}")

    # ── Percentile table: P0 → P100, step 10 ────────────────────────────────
    print("\n  Percentiles  P0 → P100  (step 10)")
    display(pct_table(lr, WIDE_PCTS, "P{}").style.format("{:.6f}"))

    # ── Percentile table: P90 → P100, step 1 ────────────────────────────────
    print("\n  Percentiles  P90 → P100  (step 1)")
    display(pct_table(lr, FINE_PCTS, "P{}").style.format("{:.6f}"))

    # ── Histogram ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(lr, bins=100, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.axvline(0, color="crimson", linewidth=1.4, linestyle="--", label="zero")
    ax.set_title(
        f"{asset.upper()} — ln(max_distant_vwap / current_vwap)  "
        f"[window = {WINDOW} samples]",
        fontsize=12,
    )
    ax.set_xlabel("log_return")
    ax.set_ylabel("count")
    ax.legend()
    plt.tight_layout()
    plt.show()